# 02 — Modeling Dataset & Leakage-Safe Split

This notebook turns the repaired data into the exact modeling table used by every model. It uses a **GroupShuffleSplit** on `vehicle_signature`, so repeated/near-identical observable vehicles do not appear in both train and test.

In [1]:
from pathlib import Path
import sys, json, os

def _find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "bonbanh_usedcar_dataset.csv").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Project root not found. Keep notebooks inside the CarValue_AI_Complete_Project folder.")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /mnt/data/CarValue_AI_Complete_Project_v2


In [2]:
import pandas as pd
import numpy as np
from src.carvalue_utils import (
    ensure_modeling_artifacts, FEATURES, CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET, get_train_test
)

model_df, reference_year = ensure_modeling_artifacts(PROJECT_ROOT, force=True)
train, test = get_train_test(model_df)

print("Reference year:", reference_year)
print("Modeling rows:", len(model_df))
print("Train rows:", len(train))
print("Test rows:", len(test))
print("Features:", FEATURES)


Reference year: 2026
Modeling rows: 23186
Train rows: 18591
Test rows: 4595
Features: ['brand', 'model', 'origin', 'body_type', 'transmission', 'fuel_type', 'exterior_color', 'drivetrain', 'car_age', 'mileage_km', 'mileage_zero_flag', 'engine_size_l', 'seats']


In [3]:
train_groups = set(train["vehicle_signature"])
test_groups = set(test["vehicle_signature"])
overlap = train_groups & test_groups

print("Unique train signatures:", len(train_groups))
print("Unique test signatures:", len(test_groups))
print("Overlapping signatures:", len(overlap))
assert len(overlap) == 0, "Leakage: a vehicle signature appears in both train and test."
assert train[TARGET].notna().all() and test[TARGET].notna().all()
assert (train[TARGET] > 0).all() and (test[TARGET] > 0).all()
print("PASS: target is valid and group split is leakage-safe.")


Unique train signatures: 14856
Unique test signatures: 3715
Overlapping signatures: 0
PASS: target is valid and group split is leakage-safe.


In [4]:
summary = pd.DataFrame({
    "dataset": ["all", "train", "test"],
    "rows": [len(model_df), len(train), len(test)],
    "median_price_million": [
        model_df[TARGET].median()/1e6, train[TARGET].median()/1e6, test[TARGET].median()/1e6
    ],
    "mean_price_million": [
        model_df[TARGET].mean()/1e6, train[TARGET].mean()/1e6, test[TARGET].mean()/1e6
    ],
})
display(summary.round(2))

out = PROJECT_ROOT / "artifacts" / "split_summary.csv"
summary.to_csv(out, index=False)
print("Saved:", out)


,dataset,rows,median_price_million,mean_price_million
0,all,23186,559.5,993.29
1,train,18591,560.0,999.84
2,test,4595,555.0,966.82


Saved: /mnt/data/CarValue_AI_Complete_Project_v2/artifacts/split_summary.csv


## Modeling rule

All subsequent notebooks load the `split` column generated here. **Do not create a different random split for another model**, otherwise model comparisons are not fair.